# Flexibility impacts in the 2030/2040 future scenarios

This notebook analyses the role of demand-side flexibility in the 2030/2040 **flex_off** and **flex_on** networks. We focus on how flexible operation of electric vehicles, thermal storage in the heating sector, and battery storage in the power sector changes demand profiles, dispatch patterns, prices, and curtailment across selected European countries. The analysis uses solved PyPSA networks and aggregates results at country level for a small set of representative countries (DE, NL, IT, PL, CZ, GR).
The notebook covers the following:

- BEVs in the transport sector.
- Thermal in the heating sector.
- Battery storage in the power sector.
- Compare locational marginal prices between flex_off and flex_on.
- Generation dispatch by country and technology.
- Analyse storage state of charge and dispatch by technology.
- Utilisation rates of fossil fuel plants.
- curtailment of wind, solar and hydro by country and technology.


## Imports

In [ ]:
import pandas as pd
import cartopy.crs as ccrs
import pypsa
import matplotlib.pyplot as plt
import numpy as np

## Load flex off and flex on solved networks 

In [ ]:
flex_off_network_path= "../../results/scenario_2030_flex_off/networks/base_s_39__3H_2030.nc"
flex_on_network_path= "../../results/scenario_2030_flex_on/networks/base_s_39__3H_2030.nc"
n_flex_off = pypsa.Network(flex_off_network_path)
n_flex_on = pypsa.Network(flex_on_network_path)

In [ ]:
high_flex_color = 'red'
low_flex_color = 'blue'

In [ ]:
other_power_consumption_techs = ["DAC", ""]

In [ ]:
def get_exogenous_power_demand(n):

    elec_load = n.loads.query("carrier == 'electricity'").index
    elec_load = list(set(n.loads_t.p_set.columns).intersection(set(elec_load)))
    demand = (
        n.loads_t.p_set[elec_load]
            .multiply(n.snapshot_weightings.objective, axis=0)
            .sum(axis=1)
    )

    return demand

def get_exogenous_power_industry_demand(n):

    elec_load = n.loads.query("carrier == 'industry electricity'").index
    demand = (n.loads.p_set[elec_load].sum())

    demand = pd.Series(
        index=n_flex_off.snapshots,
        data=demand
    ).multiply(n.snapshot_weightings.objective)

    return demand

def get_exogenous_power_EV_demand(n):
    # essentially before flex

    battery_load = n.loads.query("carrier == 'land transport EV'").index
    battery_load = list(set(n.loads_t.p_set.columns).intersection(set(battery_load)))

    charging_efficiency = n.links.query("carrier == 'BEV charger'").efficiency.mean()

    demand = (
        n.loads_t.p_set[battery_load]
            .multiply(n.snapshot_weightings.objective, axis=0)
            .sum(axis=1)
    ) / charging_efficiency
    
    return demand

def get_endogenous_power_heat_demand(n):
    # essentially after flex already

    elec_heat_techs = [c for c in n.links.carrier.unique() if 'heat' in c]
    elec_heat_techs = n.links.query("carrier in @elec_heat_techs").index

    demand = (
        n.links_t.p0[elec_heat_techs]
            .multiply(n.snapshot_weightings.generators, axis=0).sum(axis=1)
    )
    return demand

### Define countries, color_dict and carriers

In [ ]:
year=2030
countries = ['DE', 'NL', 'IT', 'PL', 'CZ', 'GR']
pypsa_to_ember = {
    "Bioenergy": "Bioenergy", "urban central solid biomass CHP": "Bioenergy",
    "urban central solid biomass CHP CC": "Bioenergy", "solid biomass": "Bioenergy",  # Added for robustness
    "gas": "Gas", "Gas": "Gas", "CCGT": "Gas", "OCGT": "Gas", "urban central gas CHP": "Gas",
    "urban central gas CHP CC": "Gas",
    "coal": "Hard coal", "Hard coal": "Hard coal", "urban central coal CHP": "Hard coal",
    "lignite": "Lignite", "Lignite": "Lignite", "urban central lignite CHP": "Lignite",
    "hydro": "Hydro", "Hydro": "Hydro", "PHS": "Hydro", "ror": "Hydro",
    "Nuclear": "Nuclear", "nuclear": "Nuclear", "uranium": "Nuclear",
    "offwind-ac": "Offshore wind", "offwind-dc": "Offshore wind",
    "offwind-float": "Offshore wind", "Offshore wind": "Offshore wind",
    "onwind": "Onshore wind", "Onshore wind": "Onshore wind",
    "oil": "Other fossil", "Other fossil": "Other fossil",
    "geothermal": "Other renewables", "Other renewables": "Other renewables",
    "solar": "Solar", "solar-hsat": "Solar", "Solar": "Solar",
    "solar rooftop": "solar rooftop",
    'battery': 'battery', 'home battery': 'battery', 'home batteries': 'battery',
    'BEV': 'BEV', 'battery EV': 'BEV', 'EV battery': 'BEV'
}
color_dict = {
    "Bioenergy": "#baa741",
    "Gas": "#e05b09",
    "Hard coal": "#545454",
    "Hydro": "#298c81",
    "Lignite": "#826837",
    "Nuclear": "#ff8c00",
    "Offshore wind": "#6895dd",
    "Onshore wind": "#235ebc",
    "Other fossil": "#000000",
    "Other renewables": "#e3d37d",
    "Solar": "#f9d002",
    "solar rooftop": "#90ee90",
    'battery': 'green', 'BEV': 'purple'
}

if 'color' in n_flex_on.carriers.columns and 'nice_name' in n_flex_on.carriers.columns:
    net_colors = n_flex_on.carriers.set_index('nice_name')['color'].to_dict()
    color_dict.update(net_colors)

## Transport sector – BEV charging

In this section, we aggregate and compare BEV charging demand time series for the flex_off and flex_on networks

In [ ]:
def demand_after_flex(n, load_carrier, charger, discharger, storage_unit):

    # get all loads of given carrier
    loads = n.loads.query("carrier in @load_carrier").index
    loads = list(set(n.loads_t.p_set.columns).intersection(set(loads)))
    
    nonflexed_demand = (
        n.loads_t.p_set[loads].multiply(n.snapshot_weightings.objective, axis=0)
            .T.groupby(n.loads.carrier).sum().T
    ).sum(axis=1)

    # constant demands are not in the time-series
    const_demand = n.loads.query("carrier in @load_carrier").p_set.sum()

    nonflexed_demand += const_demand

    # bus0 is the power bus -> charging = active power in power bus (positive means withdrawal, i.e. load)
    charger = n.links.query("carrier in @charger").index
    charging = (
        n.links_t.p0[charger]
            .multiply(n.snapshot_weightings.objective, axis=0).T
            .groupby(n.links.carrier).sum().T
    ).sum(axis=1)
    
    # bus1 in the power bus -> discharging = active power in power bus (positive means withdrawal, i.e. load)
    discharger = n.links.query("carrier in @discharger").index
    discharging = (
        n.links_t.p1[discharger]
            .multiply(n.snapshot_weightings.objective, axis=0).T
            .groupby(n.links.carrier).sum().T
    ).sum(axis=1)

    # storage units always sit at the power bus directly
    su = n.storage_units.query("carrier in @storage_unit").index
    su_p = (
        n.storage_units_t.p[su]
            .multiply(n.snapshot_weightings.objective, axis=0).T
            .groupby(n.storage_units.carrier).sum().T
    ).sum(axis=1)

    # loads of all other carriers is assumed to sit direclty at the power bus
    demand = nonflexed_demand + charging + discharging + su_p

    return demand

In [ ]:
EV_demand_off = get_exogenous_power_EV_demand(n_flex_off) / 1e3

EV_demand_on = get_exogenous_power_EV_demand(n_flex_on) / 1e3

### flex demand time series

In [ ]:
charger = ['BEV charger']
discharger = ["V2G"]
storage_unit = []

flexed_demand_high = demand_after_flex(n_flex_on, [], charger, discharger, storage_unit) / 1e3
flexed_demand_low = demand_after_flex(n_flex_off, [], charger, discharger, storage_unit) / 1e3

### Time-series plot

In [ ]:
fig,ax = plt.subplots(figsize=(12, 6))

flexed_demand_high.iloc[:168].plot(label='High Flex', color=high_flex_color, ax=ax)
flexed_demand_low.iloc[:168].plot(label='Low Flex', color=low_flex_color, ax=ax)

EV_demand_off.iloc[:168].plot(label='"Exogenous Demand"', linewidth=2.5, color='black', ax=ax)

plt.title('Resulting power demand(s) for EVs')
plt.xlabel('Time')
plt.ylabel('Demand [GW]')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
EV_battery_size = pd.Series(data=[
    n_flex_on.stores.query("carrier == 'EV battery'").e_nom.sum()/1e6,
    n_flex_off.stores.query("carrier == 'EV battery'").e_nom.sum()/1e6
], index = ["Flex High", "Flex Low"])

fig,ax = plt.subplots()

EV_battery_size.plot.bar(ax=ax)
ax.set_ylabel("Storage Capacity of EVs [GWh]")

plt.show()

## Heating sector – thermal storage 

In this section, we aggregate and compare heating demand time series for the flex_off and flex_on networks.

### Static demand 

In [ ]:
static_demand = (
    n_flex_off.loads_t.p_set.filter(like="heat")
        .T.groupby(n_flex_off.loads.carrier).sum()
        .multiply(n_flex_off.snapshot_weightings.objective).T
)

static_demand = static_demand / 1e3

### Flexed demand

In [ ]:
load_carrier = []
charger = [c for c in n_flex_on.links.carrier.unique() if 'heat' in c]

discharger = []
storage_unit = []

flexed_demand = get_endogenous_power_heat_demand(n_flex_on) / 1e3
nonflexed_demand = get_endogenous_power_heat_demand(n_flex_off) / 1e3

### Plot

In [ ]:
fig,(ax,ax1) = plt.subplots(2,1,figsize=(12, 6))

start = int(24*18)
end = int(24*24)
flexed_demand.iloc[start:end].plot(label='High Flex', color=high_flex_color, ax=ax)
nonflexed_demand.iloc[start:end].plot(label='Low Flex', color=low_flex_color, ax=ax)

static_demand.sum(axis=1).iloc[start:end].plot(label='Thermal Heat Demand', color='black', linewidth=2.5, ax=ax1)

ax.set_title('Resulting electric power demand(s)')
ax.set_ylabel('[GW]')
ax1.set_ylabel(r'[GW$_{th}$]')

ax.legend()
ax1.legend()

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))


idx = n_flex_on.stores.query("carrier in ['H2 Store']").index
(n_flex_off.stores_t.e[idx]/1e3).iloc[:].sum(axis=1).plot(color=low_flex_color, legend="Low Flex", ax=ax)
(n_flex_on.stores_t.e[idx]/1e3).iloc[:].sum(axis=1).plot(color=high_flex_color, legend="High Flex", ax=ax)

ax.set_ylabel("[GWh]")
ax.legend()

plt.show()


In [ ]:
flex = [c for c in n_flex_off.stores.carrier.unique() if "water" in c]

water_size = pd.Series(data=[
    n_flex_on.stores.query("carrier in @flex").e_nom_opt.sum()/1e6,
    n_flex_off.stores.query("carrier in @flex").e_nom_opt.sum()/1e6
], index = ["Flex High", "Flex Low"])

fig,ax = plt.subplots()

water_size.plot.bar(ax=ax)
ax.set_ylabel("Storage Capacity of Water Tanks [TWh]")

plt.show()

In [ ]:
HP = [c for c in n_flex_off.links.carrier.unique() if ("pump" in c or "resistive" in c)]

HPs = pd.Series(data=[
    n_flex_on.links.query("carrier in @HP").p_nom_opt.sum()/1e3,
    n_flex_off.links.query("carrier in @HP").p_nom_opt.sum()/1e3
], index = ["Flex High", "Flex Low"])

fig,ax = plt.subplots()

HPs.plot.bar(ax=ax)
ax.set_ylabel(r"Installed Heat Pumps or Resisitve Heaters [GW$_{el}$]")
ax.set_ylim([HPs.min()*0.9, HPs.max()*1.01])

plt.show()

## Power Sector - Battery Storage

In this section, we aggregate and compare Battery charging demand time series for the flex_off and flex_on networks

### Check for battery storage units

### Static demand time series

In [ ]:
elec_demand = (get_exogenous_power_demand(n_flex_off) + get_exogenous_power_industry_demand(n_flex_off))/1e3

In [ ]:
n_flex_on.storage_units.carrier.unique()

### Flexed demand time series

In [ ]:
flexed_demand = demand_after_flex(
    n_flex_on,
    ['electricity', 'industry electricity'],
    ['H2 Electrolysis', 'battery charger', 'home battery charger'],
    ['H2 Fuel Cell', 'H2 turbine', 'battery discharger', 'home battery discharger'],
    ['battery', 'PHS', 'hydro']
) / 1e3

nonflexed_demand = demand_after_flex(
    n_flex_off,
    ['electricity', 'industry electricity'],
    ['H2 Electrolysis', 'battery charger', 'home battery charger'],
    ['H2 Fuel Cell', 'H2 turbine', 'battery discharger', 'home battery discharger'],
    ['battery', 'PHS', 'hydro']

) / 1e3

### Plot

In [ ]:
fig,ax = plt.subplots(figsize=(12, 6))

start = 200 #int(24*30/3)
until = 400 #int(24*40/3)

flexed_demand.iloc[start:until].plot(label='High Flex', color=high_flex_color, ax=ax)
nonflexed_demand.iloc[start:until].plot(label='Low Flex', color=low_flex_color, ax=ax)
elec_demand.iloc[start:until].plot(label='Exogenous Demand', linewidth=2.5, color='black', ax=ax)
plt.title('Resulting active power demand(s) after storage, ignoring endogenous power demands [GW]')
plt.xlabel('Time')
plt.ylabel('Resulting Demand (GW)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
def plot_power_balance(n, bus_carrier = "AC"):
    # interesting bus_carriers: AC, Hydrogen Storage, low voltage
    power = n.statistics.energy_balance(aggregate_time=None).xs(bus_carrier, level=2, axis=0)
    power_pos = power.clip(lower=0)
    power_neg = power.clip(upper=0)
    
    power_pos.index = pd.MultiIndex.from_tuples(
        [(lvl0, f"{lvl1}xx") if lvl0 == "StorageUnit" else (lvl0, lvl1)
         for lvl0, lvl1 in power_pos.index],
        names=power_pos.index.names
    )
    
    power_neg.index = pd.MultiIndex.from_tuples(
        [(lvl0, f"{lvl1}yy") if lvl0 == "StorageUnit" else (lvl0, lvl1)
         for lvl0, lvl1 in power_neg.index],
        names=power_neg.index.names
    )
    
    df = pd.concat([power_pos, power_neg], axis=0)
    df = df.round()
    df = df.loc[~(df == 0).all(axis=1)]
    df = df.droplevel(0)
    
    df = df.loc[:, "2023-01-01":"2023-01-31"]/1e3   # March 2030

    fig, ax = plt.subplots(figsize=(17, 6))

    n.carriers[n.carriers.color == ''] = 'black'
    missing_carriers = list(set(df.index)-set(n.carriers.index))
    print(missing_carriers)
    carrier_map = (
        n.carriers.reset_index()
            .set_index("nice_name").Carrier.to_dict()
    )

    new_colors = {
        "Battery Storageyy": n.carriers.loc["battery", "color"],
        "Battery Storagexx": n.carriers.loc["battery", "color"],
        "Reservoir & Damxx": n.carriers.loc["hydro", "color"],
        "Pumped Hydro Storagexx": n.carriers.loc["PHS", "color"],
        "Pumped Hydro Storageyy": n.carriers.loc["PHS", "color"],
        "solar-hsat": 'yellow',
    }

    df = df[df.fillna(0).sum(axis=1).abs()>10]
    
    for missing_carrier in missing_carriers:
        add_carrier = missing_carrier
        if missing_carrier in carrier_map.keys():
            missing_carrier = carrier_map[missing_carrier]

        if missing_carrier in n.carriers.nice_name:
            color = n.carriers.loc[missing_carrier].color
        elif missing_carrier in new_colors.keys():
            color = new_colors[missing_carrier]
        else:
            print(missing_carrier)
            color = 'orange'

        n.add(
            "Carrier",
            add_carrier,
            co2_emissions=0,
            color=color,
            nice_name=missing_carrier,
            max_growth=np.inf,
            max_relative_growth=0
        )
    
    colors = n.carriers.loc[df.index].color

    df.T.plot.area(
        ax=ax,
        stacked=True,
        legend=True,
        color=colors
    )

In [ ]:
# "['Offshore Wind (AC)', 'Offshore Wind (DC)', 'Offshore Wind (Floating)', 'Onshore Wind', 'Run of River', 'Solar', 'Combined-Cycle Gas', 'Open-Cycle Gas', 'Battery Storagexx', 'Pumped Hydro Storagexx', 'Reservoir & Damxx', 'Battery Storageyy', 'Pumped Hydro Storageyy'] not in index"

In [ ]:
stores = ["H2 Store"]

store_size = pd.Series(data=[
    (
        n_flex_on.stores.query("carrier in @stores")[["e_nom", "e_nom_opt"]].max(axis=1).sum() + (
            n_flex_on.storage_units.query("carrier in @stores")[["p_nom", "p_nom_opt"]].max(axis=1)
            * n_flex_on.storage_units.query("carrier in @stores").max_hours
        ).sum()
    ) /1e3,
    (
        n_flex_off.stores.query("carrier in @stores")[["e_nom", "e_nom_opt"]].max(axis=1).sum() + (
            n_flex_off.storage_units.query("carrier in @stores")[["p_nom", "p_nom_opt"]].max(axis=1)
            * n_flex_off.storage_units.query("carrier in @stores").max_hours
        ).sum()
    ) / 1e3
], index = ["Flex High", "Flex Low"])

fig,ax = plt.subplots()

store_size.plot.bar(ax=ax)
ax.set_ylabel("Storage capacity of batteries [GWh]")
ax.set_title(f"{stores}")

plt.show()

In [ ]:
stores = ["battery", "home battery"]

store_size = pd.Series(data=[
    (
        n_flex_on.stores.query("carrier in @stores")[["e_nom", "e_nom_opt"]].max(axis=1).sum() + (
            n_flex_on.storage_units.query("carrier in @stores")[["p_nom", "p_nom_opt"]].max(axis=1)
            * n_flex_on.storage_units.query("carrier in @stores").max_hours
        ).sum()
    ) /1e3,
    (
        n_flex_off.stores.query("carrier in @stores")[["e_nom", "e_nom_opt"]].max(axis=1).sum() + (
            n_flex_off.storage_units.query("carrier in @stores")[["p_nom", "p_nom_opt"]].max(axis=1)
            * n_flex_off.storage_units.query("carrier in @stores").max_hours
        ).sum()
    ) / 1e3
], index = ["Flex High", "Flex Low"])

fig,ax = plt.subplots()

store_size.plot.bar(ax=ax)
ax.set_ylabel("Storage capacity of batteries [GWh]")
ax.set_title(f"{stores}")

plt.show()

In [ ]:
vresgens = [c for c in n_flex_off.generators.carrier.unique() if ("solar" in c and "thermal" not in c) or "wind" in c]

vrescaps = pd.Series(data=[
    n_flex_on.generators.query("carrier in @vresgens").p_nom_opt.sum()/1e3,
    n_flex_off.generators.query("carrier in @vresgens").p_nom_opt.sum()/1e3
], index = ["Flex High", "Flex Low"])

fig,ax = plt.subplots()

vrescaps.plot.bar(ax=ax)
ax.set_ylabel("VRES Capacity [GW]")

plt.show()

In [ ]:
def combine_soc_lv_and_hv(n):
    soc_s = n.stores.query("carrier in ['battery', 'home battery']").index
    soc_s = n.stores_t.e[soc_s].sum(axis=1)/1e3

    soc_su = n.storage_units.query("carrier in ['battery', 'home battery']").index
    soc_su = n.storage_units_t.state_of_charge[soc_su].sum(axis=1)/1e3

    soc = soc_s + soc_su
    return soc

In [ ]:
soc_low = combine_soc_lv_and_hv(n_flex_off)
soc_high = combine_soc_lv_and_hv(n_flex_on)

In [ ]:
fig,ax = plt.subplots(figsize=(12, 6))

soc_low.iloc[:].plot(label='Low Flex', color=low_flex_color, ax=ax)
soc_high.iloc[:].plot(label='High Flex', color=high_flex_color, ax=ax)
plt.title('Flex_off vs Flex_on Battery Storage (First Three Weeks of 2030)')
plt.xlabel('Time')
plt.ylabel('State of Charge (GW)')
plt.legend()
plt.grid(True)
plt.show()

## Electricity prices – locational marginal prices (LMPs)

In this section, we compare locational marginal prices in both networks for a selected snapshot.

In [ ]:
def get_electriciy_price_weighted(n):

    # power demand
    elec_cost_buses = n.buses.query("carrier in ['low voltage']").index
    prices = n.buses_t["marginal_price"][elec_cost_buses]
    
    elec_load = n.loads.query("bus in @elec_cost_buses").bus.values
    elec_load = list(set(elec_load).intersection(set(n.loads_t.p_set.rename(columns = n.loads.bus.to_dict()).columns)))
    loads = (
        n.loads_t.p_set
            .rename(columns = n.loads.bus.to_dict())[elec_load]
            .multiply(n.snapshot_weightings.generators, axis=0)
    )
    loads.columns.name = "Load"

    # electric industry demand
    const_loads = n.loads.p_set.rename(index=n.loads.bus.to_dict())
    const_loads = const_loads.groupby(level=0).sum()
    const_loads = pd.DataFrame(
        {bus: const_loads[bus] for bus in const_loads.index},
        index=n.snapshots
    ).rename(columns = n.loads.bus.to_dict())[elec_load].multiply(
        n.snapshot_weightings.generators, axis=0
    )
    const_loads.columns.name = "Load"

    # electric heating demand
    heat_power_tech = [c for c in n.links.carrier.unique() if 'heat' in c]
    heat_loads = n.links.query("carrier in @heat_power_tech").index
    heat_loads = (
        n.links_t.p0[heat_loads].T.groupby(n.links.bus0).sum().T
            .multiply(n.snapshot_weightings.generators, axis=0)
            .rename(columns=n.links.bus0.to_dict())[elec_load]
    )
    heat_loads.columns.name = "Load"

    # electric EV demand
    EV_tech = [c for c in n.links.carrier.unique() if 'EV charger' in c]
    EV_loads = n.links.query("carrier in @EV_tech").index
    EV_loads = (
        n.links_t.p0[EV_loads]
            .multiply(n.snapshot_weightings.generators, axis=0)
            .rename(columns=n.links.bus0.to_dict())[elec_load]
    )
    EV_loads.columns.name = "Load"

    # home battery demand
    hb_tech = [c for c in n.links.carrier.unique() if 'battery charger' in c]
    hb_loads = n.links.query("carrier in @hb_tech").index
    hb_loads = (
        n.links_t.p0[hb_loads]
            .multiply(n.snapshot_weightings.generators, axis=0)
            .rename(columns=n.links.bus0.to_dict())[elec_load]
    )
    hb_loads.columns.name = "Load"

    total_loads = (
        loads + const_loads + heat_loads + hb_loads + EV_loads
    )
    
    prices_weighted = (prices[elec_load] * total_loads).sum(axis=1) / total_loads.sum(axis=1)
    
    weighted_mean_price = prices_weighted.mean()
    
    return weighted_mean_price, prices_weighted

### Locationa Marginal Prices

In [ ]:
weighted_mean_price, flex_on_prices_weighted = get_electriciy_price_weighted(n_flex_on)
print(weighted_mean_price)
weighted_mean_price, flex_off_prices_weighted = get_electriciy_price_weighted(n_flex_off)
print(weighted_mean_price)


### plot

In [ ]:
fig, ax = plt.subplots(figsize=[19,5])

flex_on_prices_weighted.plot(ax=ax, color=high_flex_color, label="High Flex")
flex_off_prices_weighted.plot(ax=ax, color=low_flex_color, label="Low Flex")

ax.set_ylabel("Average marginal price for electricity [EUR/MW]")
ax.set_ylim([0,400])

plt.legend()

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=[15,5])

where_0 = int(24*30*1/3)
where_1 = int(24*30*2/3)

flex_on_prices_weighted.iloc[where_0:where_1].plot(ax=ax, color=high_flex_color, label="High Flex")
flex_off_prices_weighted.iloc[where_0:where_1].plot(ax=ax, color=low_flex_color, label="Low Flex")

ax.set_ylabel("Average marginal price for electricity [EUR/MW]")
ax.set_ylim([0,400])

plt.show()

## Dispatch by technology

In this section we look at how flexibility affects dispatch over time. We aggregate generator output by node and technology and plot hourly dispatch for selected nodes or countries.

In [ ]:
#n_flex_off.statistics.energy_balance(aggregate_time=None, groupby=["country", "carrier", "bus_carrier"]).query("country == 'DE'").xs("AC", level=3, axis=0).droplevel("country")

In [ ]:
n_flex_off.global_constraints

In [ ]:
n_flex_off.statistics.energy_balance(aggregate_time=None).loc["Link"]

### Compute dispatch data per country for both scenarios

In [ ]:
def plot_power_balance(n, country = None, bus_carrier = "AC"):
    if country is None:
        groupby = ["carrier", "bus_carrier"]
        power = n.statistics.energy_balance(aggregate_time=None, groupby=groupby).xs(bus_carrier, level=2, axis=0)
    else:
        country_query = "country == @country"
        groupby = ["country", "carrier", "bus_carrier"]
        power = (
            n.statistics.energy_balance(aggregate_time=None, groupby=groupby)
                .query(country_query).xs(bus_carrier, level=3, axis=0)
                .droplevel("country")
        )

    # interesting bus_carriers: AC, Hydrogen Storage, low voltage
    power_pos = power.clip(lower=0)
    power_neg = power.clip(upper=0)
    
    power_pos.index = pd.MultiIndex.from_tuples(
        [(lvl0, f"{lvl1}xx") if lvl0 == "StorageUnit" else (lvl0, lvl1)
         for lvl0, lvl1 in power_pos.index],
        names=power_pos.index.names
    )
    
    power_neg.index = pd.MultiIndex.from_tuples(
        [(lvl0, f"{lvl1}yy") if lvl0 == "StorageUnit" else (lvl0, lvl1)
         for lvl0, lvl1 in power_neg.index],
        names=power_neg.index.names
    )
    
    df = pd.concat([power_pos, power_neg], axis=0)
    df = df.round()
    df = df.loc[~(df == 0).all(axis=1)]
    df = df.droplevel(0)
    
    df = df.loc[:, "2023-01-01":"2023-01-31"]   # March 2030

    #idx = n.carriers[n.carriers.color == ''].index
    #n.carriers.loc[idx, "color"] = 'black'
    n.carriers.loc["urban central coal CHP", "color"] = n.carriers.loc["coal", "color"]
    n.carriers.loc["urban central gas CHP", "color"] = n.carriers.loc["gas", "color"]
    n.carriers.loc["urban central lignite CHP", "color"] = n.carriers.loc["lignite", "color"]
    missing_carriers = list(set(df.index)-set(n.carriers.index))
    print(missing_carriers)
    carrier_map = (
        n.carriers.reset_index()
            .set_index("nice_name").name.to_dict()
    )

    new_colors = {
        "Battery Storageyy": n.carriers.loc["battery", "color"],
        "Battery Storagexx": n.carriers.loc["battery", "color"],
        "Reservoir & Damxx": n.carriers.loc["hydro", "color"],
        "Pumped Hydro Storagexx": n.carriers.loc["PHS", "color"],
        "Pumped Hydro Storageyy": n.carriers.loc["PHS", "color"],
        "solar-hsat": 'yellow',
    }

    df = df[df.fillna(0).sum(axis=1).abs()>10]
    df = df/1e3
    
    for missing_carrier in missing_carriers:
        #print(missing_carrier)
        add_carrier = missing_carrier
        if missing_carrier in carrier_map.keys():
            missing_carrier = carrier_map[missing_carrier]

        if missing_carrier in n.carriers.nice_name:
            color = n.carriers.loc[missing_carrier].color
        elif missing_carrier in new_colors.keys():
            color = new_colors[missing_carrier]
        else:
            #print(missing_carrier)
            color = 'orange'

        n.add(
            "Carrier",
            add_carrier,
            co2_emissions=0,
            color=color,
            nice_name=missing_carrier,
            max_growth=np.inf,
            max_relative_growth=0
        )
    
    colors = n.carriers.loc[df.index].color

    if country is None:
        title = f"EU {bus_carrier} power balance"
    else:
        title = f"{country} {bus_carrier} power balance"

    fig, ax = plt.subplots(figsize=(15, 5))

    df.T.plot.area(
        ax=ax,
        stacked=True,
        legend=True,
        color=colors
    )

    plt.ylabel("GW")
    plt.title(title)

    plt.show()

In [ ]:
# interesting bus_carriers:
# AC, low voltage, gas, Hydrogen Storage

In [ ]:
plot_power_balance(n_flex_off, bus_carrier="AC")

In [ ]:
plot_power_balance(n_flex_off, bus_carrier="low voltage")

In [ ]:
plot_power_balance(n_flex_on, bus_carrier="AC")

In [ ]:
plot_power_balance(n_flex_on, bus_carrier="low voltage")

In [ ]:
n_flex_off.buses.carrier.unique()

## Utilisation rates of fossil fuel plants by node and technology

### Fossil carriers

In [ ]:
n_flex_off.statistics.energy_balance().loc["Generator"].reset_index().set_index("bus_carrier").loc[fossil_carriers][0].values

In [ ]:
n_flex_off.statistics.energy_balance().loc["Generator"].reset_index().set_index("bus_carrier").loc[fossil_carriers][0]

In [ ]:
n_flex_off.generators_t.p["EU gas"].sum()/1e6*3

In [ ]:
fossil_carriers = ["lignite", "coal", "gas", "uranium"]

fossil_utilization = pd.DataFrame(
    columns = fossil_carriers,
    index = ["Low Flex", "High Flex"],
    data = [
        n_flex_off.statistics.energy_balance().loc["Generator"].reset_index().set_index("bus_carrier").loc[fossil_carriers][0]/1e6,
        n_flex_on.statistics.energy_balance().loc["Generator"].reset_index().set_index("bus_carrier").loc[fossil_carriers][0]/1e6
    ]
)


fossil_utilization.plot.bar()
plt.ylabel(r"TWh$_{th}$")
plt.show()

### Calculate utilization

In [ ]:

def calculate_utilization(n, scenario):
    fossil_carriers_in_net = [c for c in fossil_carriers if c in n.links.carrier.unique()]
    if not fossil_carriers_in_net:
        return pd.DataFrame()
    util_df = pd.DataFrame(columns=['Node', 'Technology', 'Utilization (%)', 'Scenario'])
    # Links 
    util_links = n.statistics.capacity_factor(
        comps=['Link'],
        carrier=fossil_carriers_in_net,
        groupby=["bus1", "carrier"]
    ) * 100
    if not util_links.empty:
        util_links_df = util_links.reset_index(name='Utilization (%)')
        util_links_df['bus1'] = util_links_df['bus1'].astype(str)
        util_links_df['Node'] = util_links_df['bus1'].str[:2] 
        util_links_df['Utilization (%)'] = util_links_df['Utilization (%)'].round(2)
        util_links_df['Scenario'] = scenario
        util_links_df = util_links_df[['Node', 'carrier', 'Utilization (%)', 'Scenario']].rename(columns={'carrier': 'Technology'})
        util_df = pd.concat([util_df, util_links_df], ignore_index=True)
    util_df = util_df.groupby(['Node', 'Technology', 'Scenario'])['Utilization (%)'].mean().reset_index()
    return util_df
# Compute for both
util_on = calculate_utilization(n_flex_on, 'Flex On')
util_off = calculate_utilization(n_flex_off, 'Flex Off')
util_combined = pd.concat([util_on, util_off], ignore_index=True)
util_combined_focus = util_combined[util_combined['Node'].isin(countries)]
pivot_util = util_combined_focus.pivot(index=['Node', 'Technology'], columns='Scenario', values='Utilization (%)').fillna(0).reset_index()



### Plot

In [ ]:

fig, axs = plt.subplots(3, 2, figsize=(12, 18), sharey=True)
axs = axs.flatten()

for i, country in enumerate(["DE", "NL", "IT", "PL", "CZ", "GR"]):
    ax = axs[i]
    country_data = util_combined_focus[util_combined_focus['Node'] == country]
    if country_data.empty:
        ax.text(0.5, 0.5, f"No data for {country}", ha='center')
        continue
    
    pivot_country = country_data.pivot(index='Technology', columns='Scenario', values='Utilization (%)').fillna(0)
    
    pivot_country.plot(kind='bar', ax=ax, rot=45, width=0.8)
    ax.set_title(f"{country} Fossil Utilization Comparison")
    ax.set_ylabel('Utilization (%)')
    ax.set_xlabel('Technology')
    ax.legend(title='Scenario')
    ax.grid(True, axis='y')

plt.tight_layout()
plt.show()

## Curtailment of wind and solar and hydro by node

### definitions 

In [ ]:
countries = ['DE', 'NL', 'IT', 'PL', 'CZ', 'GR']
vres_carriers = ['onwind', 'solar', 'offwind-ac', 'ror'] 

# Carrier nice names and colors
carrier_map = {
    'onwind': 'Onshore Wind',
    'solar': 'Solar',
    'offwind-ac': 'Offshore Wind',
    'ror': 'Run-of-River Hydro'
}

color_dict = {
    'Onshore Wind': "#235ebc",
    'Offshore Wind': "#6895dd",
    'Solar': "#f9d002",
    'Run-of-River Hydro': "#298c81"
}


### calculate curtailment per scenario

In [ ]:

def calculate_curtailment(n, scenario):
    curtailment_df = pd.DataFrame(columns=['Country', 'Carrier', 'Curtailed GWh', 'Scenario'])
    for carrier in vres_carriers:
        gens = n.generators[n.generators.carrier == carrier]
        if gens.empty:
            continue
        capacity = gens.p_nom_opt if 'p_nom_opt' in gens else gens.p_nom
        p_available = n.generators_t.p_max_pu[gens.index].multiply(capacity, axis=1)
        p_dispatched = n.generators_t.p[gens.index]
        p_curtailed = p_available - p_dispatched
        p_curtailed[p_curtailed < 0] = 0
        curtailed_gwh = (p_curtailed.multiply(n.snapshot_weightings.generators, axis=0).sum().sum()) / 1e3 # MW*h to GWh
        # Group by country
        gens['Country'] = gens.bus.str[:2]
        country_curt = p_curtailed.multiply(n.snapshot_weightings.generators, axis=0).sum(axis=0).groupby(gens['Country']).sum() / 1e3
      
        for country in countries:
            if country in country_curt.index and country_curt[country] > 0:
                curtailment_df = pd.concat([curtailment_df, pd.DataFrame({
                    'Country': [country],
                    'Carrier': [carrier_map.get(carrier, carrier)],
                    'Curtailed GWh': [round(country_curt[country], 2)],
                    'Scenario': [scenario]
                })], ignore_index=True)
  
    return curtailment_df

# Compute for both
curt_on = calculate_curtailment(n_flex_on, 'Flex On')
curt_off = calculate_curtailment(n_flex_off, 'Flex Off')
curt_combined = pd.concat([curt_on, curt_off], ignore_index=True)


##  Plot

In [ ]:

def plot_country_curtailment_comparison_donut(df, country_isos, color_dict):
    n = len(country_isos)
    fig, axes = plt.subplots(n, 2, figsize=(12, 6 * n), sharex=False, sharey=False)
    plt.subplots_adjust(wspace=0.4, hspace=0.4)
    axes = axes if n == 1 else axes.flatten() 
    legend_handles = []
    legend_labels = []
    all_techs = df['Carrier'].unique() 
   
    for idx, country_iso in enumerate(country_isos):
        country_data = df[df['Country'] == country_iso]
        if country_data.empty:
            axes[idx*2].axis('off')
            axes[idx*2 + 1].axis('off')
            continue
       
        for j, scenario in enumerate(['Flex Off', 'Flex On']):
            ax = axes[idx*2 + j] if n > 1 else axes[j]
            scen_data = country_data[country_data['Scenario'] == scenario]
            if scen_data.empty:
                ax.axis('off')
                continue
           
            data = scen_data.set_index('Carrier')['Curtailed GWh']
            data = data[data > 0]
            if data.empty or data.sum() == 0:
                ax.text(0.5, 0.5, f"No Curtailment in {country_iso} ({scenario})", ha='center', va='center', fontsize=12)
                ax.axis('off')
                continue
           
            colors = [color_dict.get(tech, "#cccccc") for tech in data.index]
           
            wedges, _ = ax.pie(
                data.values,
                labels=None,
                startangle=90,
                colors=colors,
                wedgeprops=dict(width=0.7),
                autopct=None
            )
            for i, wedge in enumerate(wedges):
                angle = (wedge.theta2 + wedge.theta1) / 2
                x = 0.7 * np.cos(np.deg2rad(angle))
                y = 0.7 * np.sin(np.deg2rad(angle))
                ax.text(x, y, f"{data.values[i]:.0f}", ha='center', va='center', fontsize=10, color='white', fontweight='bold')
            centre_circle = plt.Circle((0, 0), 0.25, color='white', fc='white', linewidth=0)
            ax.add_artist(centre_circle)
           
            total = data.sum().round()
            ax.text(0, 0, f"{total:.0f}", ha='center', va='center', fontsize=14, fontweight='bold')
           
            ax.set_title(f"{country_iso} - {scenario}", fontsize=14, fontweight='bold')
           
            if not legend_handles: 
                legend_handles = wedges
                legend_labels = data.index
    for i in range(n * 2, len(axes)):
        axes[i].axis('off')
   
    fig.legend(
        legend_handles, legend_labels,
        loc='upper center',
        bbox_to_anchor=(0.5, 1.02),
        ncol=4,
        fontsize=10,
        frameon=True
    )
   
    fig.suptitle("Yearly Curtailment Comparison [GWh] (PyPSA-Eur)", fontsize=16, weight='bold', y=1.05)
    plt.tight_layout(rect=[0, 0, 1, 0.98])
    plt.show()

plot_country_curtailment_comparison_donut(curt_combined, countries, color_dict)

In [ ]:
country = "GR"
tech = "onwind"

m = n_flex_on.copy()

avail = m.generators_t.p_max_pu.multiply(m.generators.p_nom).filter(like=country).filter(like=tech).multiply(m.snapshot_weightings.objective,axis=0).sum()/1e3
disp = m.generators_t.p.filter(like=country).filter(like=tech).multiply(m.snapshot_weightings.objective,axis=0).sum()/1e3

avail-disp